# 📝 과제 LV3(통합): 멀티페이지 AI 대시보드

데이터 대시보드 + 챗봇 + 문서 Q&A 를 **하나의 멀티페이지 앱**으로 묶고, 기존 시스템(`core`)을 실제로 연동합니다.

| | |
| --- | --- |
| 데이터 | `data/diamonds.csv` (다이아몬드 5,000개: 캐럿·컷·색상·투명도·가격) |
| 실행 | 루트에서 `uv run streamlit run 과제_LV3_통합/main.py` |
| 연동 대상 | `core.chatbot_core` · `core.rag_core` (재작성 금지) |
| 키 | `.streamlit/secrets.toml` 의 `OPENAI_API_KEY` (없으면 챗봇·문서 Q&A 는 안내를 띄우고 멈춥니다) |

```text
main.py               # 엔트리(네비게이션)
pages/1_홈.py
pages/2_대시보드.py    # 다이아몬드
pages/3_챗봇.py        # core.chatbot_core
pages/4_문서QA.py      # core.rag_core
```

**전체 규칙**: 각 페이지는 **자립적**이어야 합니다. 필요한 데이터·상태는 그 페이지 안에서 준비하세요. 페이지를 넘나드는 값만 `st.session_state` 에 둡니다(이 과제에서는 **대화 이력**과 **대시보드 필터** 둘뿐입니다).

### LV2 에서 무엇이 늘어나는가

이 과제는 LV2 에서 만든 것을 **그대로 가져와 한 겹씩 얹는** 구조입니다. 새로 배우는 것은 오른쪽 칸뿐입니다.

| LV2 에서 한 것 | LV3 에서 더하는 것 |
| --- | --- |
| 탭 3개로 한 화면에 담기(틀은 제공받음) | `st.navigation` 으로 **멀티페이지 직접 구성** (문제 1) |
| 챗봇: 이력·입력·스트리밍 | + **대시보드 필터를 질문에 실어 보내기** (2-챗봇) |
| 문서 Q&A: `ask` + 근거 `expander` | + **`@st.cache_resource` 로 검색 자원 재사용** (2-문서QA) |
| 대시보드: 캐싱 로딩 + 지표 + plotly | + **필터**, + 탭 3개(plotly · seaborn · 표) (2-대시보드) |
| (없음) | **페이지 간 상태 공유**: 대화 이력과 필터 (2-홈) |
| (없음) | **배포 의존성 도출** (문제 3) |

데이터는 레벨마다 바뀝니다(LV1 타이타닉 → LV2 택시 → LV3 다이아몬드). 같은 기능을 **처음 보는 데이터에 적용**하는 것도 이 과제의 일부입니다.


### 완성 화면

**🏠 홈** / **💎 대시보드**

![LV3 홈](../images/lv3/lv3_home.png)

![LV3 대시보드](../images/lv3/lv3_dashboard.png)

**💬 챗봇** / **📚 문서 Q&A**

![LV3 챗봇](../images/lv3/lv3_chat.png)

![LV3 문서 Q&A](../images/lv3/lv3_rag.png)


## 1. 멀티페이지 골격 (`main.py`)

- `st.navigation` 에 **딕셔너리**를 넘겨 그룹을 나눈다: `"시작"`(홈) · `"분석"`(대시보드) · `"AI"`(챗봇·문서 Q&A)
- 각 페이지는 `st.Page("pages/파일명.py", title=..., icon=...)` 로 등록하고, 홈에만 `default=True`
- 반환값을 `pg` 에 담고 마지막 줄에서 `pg.run()` (이 줄이 없으면 빈 화면만 뜹니다)

**확인**: 사이드바에 시작/분석/AI 그룹과 메뉴 4개가 보이고, 클릭하면 화면이 바뀐다.


## 2-홈 (`pages/1_홈.py`)

- `st.markdown` 으로 앱 소개 문구
- 페이지 간 공유 상태 초기화: `st.session_state.messages` 를 빈 리스트로. **키가 없을 때만**
- `st.info` 로 사이드바 이동 안내

**확인**: 앱을 처음 켠 직후에도 챗봇 페이지가 오류 없이 열린다.


## 2-대시보드 (`pages/2_대시보드.py`)

- `@st.cache_data` 로 `diamonds.csv` 로드 (제공된 `DATA_DIR` 사용)
- 사이드바 필터: 컷 등급(`st.multiselect`) · 최대 가격(`st.slider`)
- 지표(`st.columns` + `st.metric`) 3개: 개수 · 평균 가격(`$` 천단위) · 평균 캐럿(소수 2자리)
- `st.tabs` 로 탭 3개
  - **컷별 개수**: `px.bar` → `st.plotly_chart`
  - **캐럿-가격 관계**: seaborn `scatterplot` → `st.pyplot(fig)` → `plt.close(fig)`
  - **데이터**: `st.dataframe` 으로 필터된 표
- 고른 필터를 `st.session_state` 에 저장한다. 예: `dashboard_filter` 키에 컷 목록과 최대 가격을 담은 딕셔너리 (**챗봇 페이지가 이 값을 읽어 갑니다**)

**확인**: 컷을 "Ideal" 만 남기고 최대 가격을 낮추면 지표·차트·표가 모두 그에 맞게 바뀐다.


## 2-챗봇 (`pages/3_챗봇.py`)

- 공유 `messages` 이력 준비 → 사이드바 "대화 초기화" 버튼 → 이력 다시 그리기
- 새 입력을 `chatbot_core.stream_reply` 로 스트리밍 (이력에서 **방금 추가한 내 메시지는 제외**)
- 여기까지의 위젯 구성은 **LV2 문제 1~3과 같습니다**. 아래 항목이 이 레벨에서 새로 추가되는 부분입니다
- **대시보드 필터를 질문에 실어 보낸다. 이 레벨의 핵심입니다.**
  - 화면 위쪽에 지금 적용 중인 조건을 `st.caption` 으로 표시
  - 모델에 보낼 때는 그 조건을 설명하는 문단을 **질문 앞에 덧붙여** 첫 번째 인자로 넘긴다
  - **말풍선과 이력에 남는 것은 학생이 입력한 원래 문장 그대로**여야 한다 (조건 문단은 화면에 노출하지 않는다)
  - 대시보드를 한 번도 열지 않았으면 공유 상태가 없다. 그때는 조건 없이 평소처럼 답한다 (`st.session_state` 를 무조건 읽으면 첫 실행에서 앱이 죽습니다)

**확인**: 대시보드에서 조건을 바꾼 뒤 챗봇에서 "이 조건이면 평균 가격이 왜 이렇게 나올까?" 라고 물으면 **모델이 그 조건을 이미 알고** 답한다. 말풍선에는 내가 친 문장만 보인다.


## 2-문서 Q&A (`pages/4_문서QA.py`)

- 검색 자원은 `@st.cache_resource` 로 **한 번만** 준비해 재사용 (재실행마다 인덱스를 새로 만들면 느려집니다)
- 사이드바 참고 문서 수(`st.slider`) → 질문(`st.text_input`)
- `rag_core.ask` 로 답변 표시 + 근거 문서를 `st.expander` 로

**확인**: 답변과 함께 근거 문서가 접힌 목록으로 나오고, 두 번째 질문부터는 첫 응답보다 빠르다.


## 3. 배포 준비: 의존성 목록 도출

배포 서버는 저장소의 **의존성 파일 하나**만 보고 환경을 만듭니다. 한 줄이라도 빠지면 배포는 `ModuleNotFoundError` 로 죽습니다.

**함정**: 대부분 **자기가 쓴 `import` 문만** 적습니다. 하지만 `pages/3_챗봇.py` 는 `core.chatbot_core` 를 부르고, 그 안에서 또 다른 라이브러리를 씁니다. **내가 직접 import 하지 않아도 앱이 도는 데 필요한 것은 전부** 목록에 있어야 합니다.

**요구사항**

- `과제_LV3_통합/` 에서 `uv init . -p 3.12` 로 프로젝트를 만들고, 도출한 목록을 `uv add` 로 추가한다
- 결과로 `pyproject.toml`(무엇이 필요한지)과 `uv.lock`(어떤 버전으로)이 생긴다. **둘 다** 배포 대상이다
- 저장소에 의존성 파일은 **하나만** 둔다 (여러 개면 먼저 만난 하나만 읽힙니다)
- 완성본 예시는 `교안_02_챗봇과_연동/04_멀티페이지_데모/pyproject.toml` 에 있다. 먼저 스스로 도출한 뒤 비교할 것

**도출 절차**

1. `main.py` 와 4개 페이지의 import 를 모은다
2. 그 페이지들이 부르는 `core/` 모듈(`chatbot_core`·`rag_core`·`keys`·`fonts`)의 import 를 더한다. **함수 안에 들어 있는 지연 import 도 포함**
3. 표준 라이브러리(`os`·`sys`·`pathlib`·`functools`·`typing`·`tomllib`)와 우리 코드(`core`)는 뺀다
4. 설치 이름과 import 이름이 다른 것에 주의한다 (`langchain_core` → `langchain-core`)

**합격 기준**

- `dependencies` 에 서드파티 패키지가 **9개 안팎** 들어간다 (직접 import 5종 + `core` 를 통해 필요한 것 4종)
- 목록의 각 패키지를 실제로 import 해 전부 오류가 없다
- 앱을 다시 실행해 4개 페이지를 모두 눌렀을 때 챗봇·문서 Q&A 까지 실제 답변이 나온다
